# Quantum Optimization of Construction Scheduling — Google Colab (GPU)

Runs the full pipeline on **real construction projects** (DSLIB v3.4, Ghent University) and uses Colab's
**GPU** for the quantum (QAOA) simulation, which allows a larger quantum core of up to **26 qubits**
(a laptop CPU handles up to 16 qubits).

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`. Then run the cells top to bottom.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > T4 GPU"

## 2. Upload the project zip
Click **Choose Files** and select the project zip you downloaded.

In [ ]:
from google.colab import files
import os, zipfile
up = files.upload()
name = next(iter(up))
zipfile.ZipFile(name).extractall("/content")
os.chdir("/content/construction_scheduling")
print("Working in", os.getcwd()); print(os.listdir("."))

## 3. Install packages (≈2 minutes)
`qiskit-aer-gpu` is the GPU build of the same simulator the laptop uses, so the CPU build is removed first.

In [ ]:
!pip -q install -r requirements.txt
!pip -q uninstall -y qiskit-aer
!pip -q install qiskit-aer-gpu

In [ ]:
from qiskit_aer import AerSimulator
print("Simulator devices:", AerSimulator().available_devices())   # expect ('CPU', 'GPU')

## 4. Run the whole pipeline
With a GPU present the demo automatically uses the **26-qubit** quantum core (the largest conflict core that fits). Expect a few minutes, most of it QAOA tuning.

In [ ]:
!python run_demo.py

## 5. Run the automated tests

In [ ]:
!python -m pytest -q -p no:cacheprovider

## 6. Look at the charts

In [ ]:
from IPython.display import Image, display
import glob, os
for f in ["results/C2025-05_bottlenecks.png", "results/decomposition_fixed_vs_adaptive.png",
          "results/C2025-05_gantt_hybrid.png", "results/C2025-05_resources_hybrid.png"] + sorted(glob.glob("results/core_*qaoa_vs_random.png")):
    if os.path.exists(f):
        print(f); display(Image(f, width=900))

## 7. Download all results

In [ ]:
import shutil
shutil.make_archive("/content/results", "zip", "results")
files.download("/content/results.zip")

## Optional: real IBM quantum computer
Needs the free IBM Open Plan account (see README §4). Replace the two values, run once, then run the next cell.

In [ ]:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(token="YOUR_API_KEY", instance="YOUR_CRN", set_as_default=True, overwrite=True)

In [ ]:
# !python run_ibm.py --fake     # rehearsal on a noisy replica of ibm_kingston, no account needed
# !python run_ibm.py            # the real ibm_kingston